# EXTRACCIÓN DE DATA WATCHMODE

### 1. Importaciones

In [1]:
import sys
import json
import os
import requests
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

### 2. Configuración del proyecto

In [3]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import (
    BRONZE_MOVIELENS_DIR,
    BRONZE_WATCHMODE_DIR
)

In [4]:
print("Proyecto:", PROJECT_ROOT)
print("MovieLens:", BRONZE_MOVIELENS_DIR)
print("Watchmode:", BRONZE_WATCHMODE_DIR)

Proyecto: d:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch
MovieLens: D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\bronze\movielens
Watchmode: D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\bronze\watchmode


### 3. Credenciales Watchmode

In [5]:
load_dotenv(PROJECT_ROOT / ".env")

WATCHMODE_API_KEY = os.getenv("WATCHMODE_API_KEY")

if not WATCHMODE_API_KEY:
    raise ValueError("No se encontró WATCHMODE_API_KEY en el archivo .env")

### 4. Leer IDs TMDb desde MovieLens

In [6]:
ruta_movielens = BRONZE_MOVIELENS_DIR / "ml-32m"

links = pd.read_csv(
    ruta_movielens / "links.csv"
)

links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [7]:
tmdb_ids = (
    links["tmdbId"]
    .dropna()
    .astype(int)
    .unique()
)

print("IDs TMDb disponibles:", len(tmdb_ids))

IDs TMDb disponibles: 87425


### 5. Headers de Watchmode

In [8]:
headers = {
    "X-API-Key": WATCHMODE_API_KEY,
    "accept": "application/json"
}

### 6. Probar una película

In [ ]:
tmdb_id = tmdb_ids[0]

watchmode_id = f"movie-{tmdb_id}"

url = (
    f"https://api.watchmode.com/v1/title/"
    f"{watchmode_id}/sources/"
)

params = {
    "regions": "PE"
}

' params = {\n    "regions": "PE"\n} '

Para disponibilidad en Perú

In [10]:
respuesta = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

respuesta.raise_for_status()

fuentes = respuesta.json()

In [11]:
print(fuentes)

[{'source_id': 372, 'name': 'Disney+', 'type': 'sub', 'region': 'PE', 'ios_url': 'Deeplinks available for paid plans only.', 'android_url': 'Deeplinks available for paid plans only.', 'web_url': 'https://www.disneyplus.com/browse/entity-f6174ebf-cb92-453c-a52b-62bb3576e402', 'format': '4K', 'price': None, 'seasons': 0, 'episodes': 0}]


### 7. Guardar respuesta cruda en Bronze

In [12]:
archivo_salida = (
    BRONZE_WATCHMODE_DIR
    / f"movie_{tmdb_id}_sources.json"
)

with open(
    archivo_salida,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        fuentes,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print("Guardado en:", archivo_salida)

Guardado en: D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\bronze\watchmode\movie_862_sources.json


### 8. Probar con 10 películas

In [13]:
ids_prueba = tmdb_ids[:10]

for tmdb_id in ids_prueba:

    archivo_salida = (
        BRONZE_WATCHMODE_DIR
        / f"movie_{tmdb_id}_sources.json"
    )

    if archivo_salida.exists():
        print(f"{tmdb_id}: ya existe.")
        continue

    watchmode_id = f"movie-{tmdb_id}"

    url = (
        f"https://api.watchmode.com/v1/title/"
        f"{watchmode_id}/sources/"
    )

    params = {
        "regions": "PE"
    }

    try:
        respuesta = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30
        )

        respuesta.raise_for_status()

        datos = respuesta.json()

        with open(
            archivo_salida,
            "w",
            encoding="utf-8"
        ) as archivo:
            json.dump(
                datos,
                archivo,
                ensure_ascii=False,
                indent=2
            )

        print(f"{tmdb_id}: OK")

    except requests.RequestException as error:
        print(f"{tmdb_id}: ERROR - {error}")

862: ya existe.
8844: OK
15602: OK
31357: OK
11862: OK
949: OK
11860: OK
45325: OK
9091: OK
710: OK


### 9. Validar archivos descargados

In [14]:
archivos_watchmode = list(
    BRONZE_WATCHMODE_DIR.glob(
        "movie_*_sources.json"
    )
)

print(
    "Archivos Watchmode descargados:",
    len(archivos_watchmode)
)

Archivos Watchmode descargados: 10


### 10. Validar tamaños

In [15]:
for archivo in archivos_watchmode[:10]:

    tamano_kb = archivo.stat().st_size / 1024

    print(
        archivo.name,
        "->",
        round(tamano_kb, 2),
        "KB"
    )

movie_11860_sources.json -> 0.0 KB
movie_11862_sources.json -> 0.35 KB
movie_15602_sources.json -> 0.0 KB
movie_31357_sources.json -> 0.35 KB
movie_45325_sources.json -> 0.39 KB
movie_710_sources.json -> 0.0 KB
movie_862_sources.json -> 0.39 KB
movie_8844_sources.json -> 0.0 KB
movie_9091_sources.json -> 0.0 KB
movie_949_sources.json -> 0.36 KB


### 11. Validar un archivo con datos

In [26]:
archivo_con_datos = None
ejemplo = None

for archivo in archivos_watchmode:
    with open(
        archivo,
        "r",
        encoding="utf-8"
    ) as f:
        datos = json.load(f)

    if datos:
        archivo_con_datos = archivo
        ejemplo = datos
        break

if ejemplo:
    print("Archivo con disponibilidad:", archivo_con_datos.name)
    print("Fuentes encontradas:", len(ejemplo))
else:
    print("No se encontraron archivos con disponibilidad.")

Archivo con disponibilidad: movie_11862_sources.json
Fuentes encontradas: 1


### 12. Manejo de respuestas vacías

In [27]:
for archivo in archivos_watchmode:
    with open(
        archivo,
        "r",
        encoding="utf-8"
    ) as f:
        datos = json.load(f)

    if datos:
        print(
            archivo.name,
            "->",
            len(datos),
            "fuentes encontradas"
        )
    else:
        print(
            archivo.name,
            "-> sin disponibilidad en la región consultada"
        )

movie_11860_sources.json -> sin disponibilidad en la región consultada
movie_11862_sources.json -> 1 fuentes encontradas
movie_15602_sources.json -> sin disponibilidad en la región consultada
movie_31357_sources.json -> 1 fuentes encontradas
movie_45325_sources.json -> 1 fuentes encontradas
movie_710_sources.json -> sin disponibilidad en la región consultada
movie_862_sources.json -> 1 fuentes encontradas
movie_8844_sources.json -> sin disponibilidad en la región consultada
movie_9091_sources.json -> sin disponibilidad en la región consultada
movie_949_sources.json -> 1 fuentes encontradas


In [29]:
con_disponibilidad = 0
sin_disponibilidad = 0

for archivo in archivos_watchmode:
    with open(
        archivo,
        "r",
        encoding="utf-8"
    ) as f:
        datos = json.load(f)

    if datos:
        con_disponibilidad += 1
    else:
        sin_disponibilidad += 1

print("Con disponibilidad:", con_disponibilidad)
print("Sin disponibilidad:", sin_disponibilidad)
print("Total revisados:", len(archivos_watchmode))

Con disponibilidad: 5
Sin disponibilidad: 5
Total revisados: 10
